In [ ]:
import os



In [ ]:
from Datasets.utilities import *
from Datasets.wiki_notable.notable import WikiNotable
os.environ['NationalityDatasets'] = "./data/"
notable = WikiNotable()

### Reading the raw data

In [ ]:
data = notable.get_raw_data()
data.shape

In [ ]:
data.head()

In [ ]:
data = data[['name', 'gender', 'all_geography_groups', 'string_citizenship_raw_d', 'citizenship_1_b', 'area1_of_rattachment', 'un_subregion']]

### Exploring the data

In [ ]:
data.sample(n=10)

In [ ]:
data['citizenship_1_b'].value_counts()

In [ ]:
data['all_geography_groups'].value_counts()

In [ ]:
data['un_subregion'].value_counts()

##### Checking None values

In [ ]:
data.fillna('', inplace=True)

In [ ]:
data[data.name=='']

In [ ]:
data.name = data.name.apply(lambda x: " ".join(x.split('_')))

In [ ]:
data[data.all_geography_groups==''].head(5)

In [ ]:
data[(data['citizenship_1_b'] == 'US') & (data['all_geography_groups'] != "US") & (data['all_geography_groups'] != "")].head(5)

The 'all_geography_groups' column is more close to our intention than citizenship! so we will work on this column as the nationality or country of people

In [ ]:
data[(data['all_geography_groups'] == '')]

In [ ]:
data[(data['all_geography_groups'] == '') & (data['citizenship_1_b'] == '')].area1_of_rattachment.value_counts()

Here, we replace and use the citizenship for those who don't have all geography groups!

In [ ]:
data['all_geography_groups'] = data.apply(lambda x: x.citizenship_1_b if x.all_geography_groups == '' else x.all_geography_groups, axis=1)

In [ ]:
print(data.shape)
data = data[data['all_geography_groups'] != '']
print(data.shape)

We check those rows that have more than one country in their all geography groups

In [ ]:

data['has_one_geography_group'] = data['all_geography_groups'].apply(lambda x: True if (',' not in x) and (x != '' ) else False)
data[data['has_one_geography_group'] == False]

In [ ]:
data[data['has_one_geography_group'] == False].all_geography_groups.value_counts()

In [ ]:
data['all_geography_groups'] = data['all_geography_groups'].apply(lambda x: x.replace('Old_regimes_in_/_of_', '').replace('United_Kingdom_of_Great_Britain_and_Northern_Ireland', 'United Kingdom').replace('Dominion_of_', '').replace('Former_Soviet_Union', 'Russia'))
data['all_geography_groups'] = data['all_geography_groups'].apply(lambda x: x.replace('US', 'United Kingdom').replace('Australia','United Kingdom').replace('Ireland', 'United Kingdom').replace('Dominion_of_', '').replace('Ukraine', 'Russia').replace('Bangladesh', 'Pakistan') if ',' in x else x)
data['all_geography_groups'] = data['all_geography_groups'].apply(lambda x: x.split(',')[0] if (',' in x) and (x.split(',')[0] == x.split(',')[1]) else x)

We delete those rows that have more than one country in their all geography groups

In [ ]:
data['has_one_geography_group'] = data['all_geography_groups'].apply(lambda x: True if (',' not in x) and (x != '' ) else False)
data[data['has_one_geography_group'] == False]

In [ ]:
print(data.shape)
data = data[data['has_one_geography_group'] == True].drop(columns='has_one_geography_group')
print(data.shape)

Some preprocessing to find right country for a person! since the citizenship value is cleaner than all_geography_group, here we use it if it has occured in the all_geography groups. For example, if the all_geography_group is 'Old_regimes_in_/_of_Germany' and citizenship is 'Germany', we keep citizenship instead of all_geography_groups!  Not necessary cell :)

In [ ]:

data['country'] = data.apply(lambda x: x['citizenship_1_b'] if x['citizenship_1_b'] in x['all_geography_groups'] else x['all_geography_groups'], axis=1)

In [ ]:
data.country.value_counts()

###### Weird country names!

In [ ]:
for i in data.country.value_counts()[data.country.value_counts()>100].index:
    print(i, end=', ')

In [ ]:
country_name_correcting = {
    'Former_Soviet_Union': 'Russia',
    'Holy_Roman_Empire': 'Italy',
    'United_Kingdom_of_Great_Britain_and_Northern_Ireland': 'United_Kingdom',
    'Korean_Peninsula': 'Korea',
    'Swaziland': 'Eswatini',
    'American_Samoa': 'US',
    'Old_regimes_in_/_of_Russia': 'Russia',
    'Old_regimes_in_/_of_Turkey': 'Turkey',
    'Federated_States_of_Micronesia': 'Micronesia',
    'United_States_Virgin_Islands': 'US',
    'Old_regimes_in_/_of_India': 'India',
    'Old_regimes_in_/_of_Austria': 'Austria',
    'British_North_America': 'Canada',
    'British_Empire': 'United_Kingdom',
    'Old_regimes_in_/_of_Korean_Peninsula': 'Korea',
    'Old_regimes_in_/_of_Romania': 'Romania',
    'Republic_of_Siena': 'Italy',
    'Duchy_of_Mantua': 'Italy',
    'Old_regimes_in_/_of_Italy': 'Italy',
    'Old_regimes_in_/_of_Spain': 'Spain',
    'Old_regimes_in_/_of_China': 'China',
    'Roman_Empire': 'Italy',
    'Abkhazia': 'Georgia',
    'Ossetia': 'Georgia',
}
data.country = data.country.apply(lambda x: country_name_correcting[x] if x in country_name_correcting else x)
data.country = data.country.apply(lambda x: " ".join(x.split('_')))

In [ ]:
data['standard country name'] = data.country.apply(lambda x: country_name_standardizer(x))

In [ ]:
data[(data['standard country name']=='') & (data.country != '')].country.value_counts()

In [ ]:
data.country = data['standard country name']
data.country.value_counts()

In [ ]:
data[data.country == '']

In [ ]:
print(data.shape)
data = data[data.country != '']
print(data.shape) 

##### Checking duplicate values

In [ ]:
data.name.value_counts()[data.name.value_counts()> 2]

In [ ]:
data[data.name=='Harald Hansen']

In [ ]:
data[data.duplicated(subset=['name', 'country'], keep=False)]

In [ ]:
print(data.shape)
data.drop_duplicates(subset=['name', 'country'], keep='first', inplace=True)
print(data.shape)

In [ ]:
data.name.value_counts()[data.name.value_counts()>=2]

In [ ]:
data[data.name == 'Paul Metz']

In [ ]:
# print(data.shape)
# data.drop_duplicates(subset=['name'], keep=False, inplace=True)
# print(data.shape)

### Saving the data

In [ ]:
data['id'] = data.apply(lambda x:'notable-'+str(x.name), axis=1)


In [ ]:
data.sample(n=10)

In [ ]:
data['full-name'] = data['name']

In [ ]:
processed_data = data[['id', 'full-name', 'country']]
processed_data.set_index('id', inplace=True)

In [ ]:
notable.save_data(processed_data)

In [ ]:
notable.data = processed_data

In [ ]:
notable.save_split_data()

In [ ]:
processed_data.shape

In [ ]:
notable.get_data_path()